In [1]:
import jax.numpy as jnp
import sax
import sax.models as sm



In [ ]:
wls = jnp.linspace(1.5, 1.6, 200)
neffs = jnp.asarray([2.39852119 2.39788955 2.39725789 2.39662621 2.39599451 2.3953628
 2.39473106 2.39409931 2.39346754 2.39283575 2.39220394 2.39157212
 2.39094027 2.39030841 2.38967653 2.38904463 2.38841272 2.38778078
 2.38714883 2.38651687 2.38588488 2.38525288 2.38462087 2.38398883
 2.38335678 2.38272471 2.38209263 2.38146053 2.38082842 2.38019629
 2.37956414 2.37893198 2.3782998  2.37766761 2.3770354  2.37640318
 2.37577094 2.37513869 2.37450642 2.37387414 2.37324184 2.37260953
 2.37197721 2.37134487 2.37071252 2.37008016 2.36944778 2.36881539
 2.36818298 2.36755057 2.36691814 2.36628569 2.36565324 2.36502077
 2.36438829 2.3637558  2.36312329 2.36249078 2.36185825 2.36122571
 2.36059316 2.3599606  2.35932803 2.35869544 2.35806285 2.35743024
 2.35679763 2.356165   2.35553237 2.35489972 2.35426707 2.3536344
 2.35300173 2.35236905 2.35173636 2.35110365 2.35047094 2.34983823
 2.3492055  2.34857276 2.34794002 2.34730727 2.34667451 2.34604174
 2.34540897 2.34477619 2.3441434  2.3435106  2.3428778  2.34224499
 2.34161218 2.34097935 2.34034653 2.33971369 2.33908085 2.33844801
 2.33781516 2.3371823  2.33654944 2.33591658 2.33528371 2.33465083
 2.33401795 2.33338507 2.33275218 2.33211929 2.3314864  2.3308535
 2.3302206  2.3295877  2.32895479 2.32832188 2.32768897 2.32705605
 2.32642314 2.32579022 2.3251573  2.32452438 2.32389145 2.32325853
 2.3226256  2.32199268 2.32135975 2.32072683 2.3200939  2.31946097
 2.31882804 2.31819512 2.31756219 2.31692927 2.31629634 2.31566342
 2.3150305  2.31439758 2.31376466 2.31313174 2.31249883 2.31186592
 2.31123301 2.3106001  2.3099672  2.3093343  2.3087014  2.30806851
 2.30743562 2.30680273 2.30616985 2.30553698 2.3049041  2.30427124
 2.30363838 2.30300552 2.30237267 2.30173982 2.30110698 2.30047415
 2.29984132 2.2992085  2.29857569 2.29794288 2.29731008 2.29667729
 2.2960445  2.29541173 2.29477896 2.2941462  2.29351345 2.2928807
 2.29224797 2.29161524 2.29098253 2.29034982 2.28971713 2.28908444
 2.28845176 2.2878191  2.28718645 2.2865538  2.28592117 2.28528855
 2.28465594 2.28402334 2.28339076 2.28275819 2.28212563 2.28149308
 2.28086055 2.28022803 2.27959552 2.27896303 2.27833055 2.27769808
 2.27706563 2.2764332  2.27580078 2.27516837 2.27453598 2.27390361
 2.27327125 2.27263891])

def interp_neff(wl=1.55):
    """
    Interpola neff(wl) siguiendo el ejemplo oficial de SAX:
    se interpola frente a 1 / wl, no directamente frente a wl.

    wl en micras.
    """
    wl = jnp.asarray(wl)

    return jnp.interp(
        1 / wl,
        1 / wls[::-1],
        neffs[::-1],
    )



# MODELS: 

https://flaport.github.io/sax/models/#sax.models.mmi2x2 


### WAVEGUIDE

In [ ]:
def waveguide(wl=1.55, wl0=1.55, neff=2.34, ng=3.4, length=10.0, loss=0.0) -> sax.SDict:
    dwl = wl - wl0
    dneff_dwl = (ng - neff) / wl0
    neff = neff - dwl * dneff_dwl
    phase = 2 * jnp.pi * neff * length / wl
    transmission = 10 ** (-loss * length / 20) * jnp.exp(1j * phase)
    sdict = sax.reciprocal(
        {
            ("in0", "out0"): transmission,
        }
    )
    return sdict

### WAVEGUIDE (SPIRAL)

In [ ]:
def straight(
    *,
    wl=1.55,
    length=10.0,
    loss=0.0,
):
    """
    Guía de onda con neff dependiente de longitud de onda,
    siguiendo el modelo de SAX.

    wl     : longitud de onda en micras
    length : longitud de la guía en micras
    loss   : pérdida en dB por micra, si se usa exactamente como en SAX

    Nota:
    En el ejemplo de SAX, la amplitud se calcula como:
        10 ** (-loss * length / 20)
    Por tanto, para no modificar su modelo, loss debe estar en la misma
    unidad inversa que length.
    """
    neff = interp_neff(wl)

    phase = 2 * jnp.pi * neff * length / wl
    amplitude = jnp.asarray(10 ** (-loss * length / 20), dtype=complex)

    transmission = amplitude * jnp.exp(1j * phase)

    return sax.reciprocal({
        ("in0", "out0"): transmission,
    })

### MMI 2x2

In [2]:
def coupler(coupling=0.5) -> sax.SDict:
    kappa = coupling**0.5
    tau = (1 - coupling) ** 0.5
    coupler_dict = sax.reciprocal(
        {
            ("in0", "out0"): tau,
            ("in0", "out1"): kappa * np.exp(-1j * np.pi / 2.0),
            ("in1", "out0"): kappa * np.exp(-1j * np.pi / 2.0),
            ("in1", "out1"): tau,
        }
    )
    return coupler_dict

### MMI 3X3